# OpenEnv-DDI GRPO Training (Remote HF Space, No Local Imports)

This notebook is designed to run on a different server and connect to a deployed OpenEnv environment via an HF Space URL.

Flow:
1. Configure remote run arguments
2. Define self-contained env client + helper functions
3. Define rollout + reward functions
4. Initialize model, trainer, and dataset
5. Train and save artifacts

In [ ]:
    # If needed, install training/runtime dependencies first:
    # !uv pip install -U "trl" "unsloth" "transformers" "datasets" "requests"

    from __future__ import annotations

    import argparse
    from dataclasses import dataclass, field
    import json
    from pathlib import Path
    import re
    from typing import Any, Callable, Optional, Protocol

    import requests
    from datasets import Dataset, load_dataset
    from transformers import AutoTokenizer

    DEFAULT_DATASET_PROMPT = "Perform safe DDI triage with strict JSON action output."
    LORA_TARGET_MODULES = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]
    HARD_REGIMEN_DELTA_THRESHOLD = 0.5
    JSON_BLOCK = re.compile(r"\{.*\}", re.DOTALL)

    SYSTEM_PROMPT = (
        "You are a clinical medication safety triage assistant. "
        "Return exactly one valid JSON object and nothing else. "
        "Required keys: action_type, interaction_id, suggested_regimen_id, rationale. "
        "Allowed action_type values: flag_interaction, monitor, suggest_alternative, ignore, finish. "
        "For flag_interaction/monitor/ignore provide interaction_id and set suggested_regimen_id to null. "
        "For suggest_alternative provide suggested_regimen_id and set interaction_id to null. "
        "For finish set both IDs to null."
    )

## 1. Configure Arguments (Remote Env)

Set env_base_url to your deployed HF Space endpoint (required).

In [ ]:
args = argparse.Namespace(
    model_name="Qwen/Qwen2.5-14B-Instruct",
    sft_dataset=Path("training/data/sft_warmstart_train.jsonl"),
    output_dir=Path("training/outputs/qwen25-14b-ddi-grpo-notebook"),
    train_steps=120,
    warmup_steps=30,
    max_turns=12,
    dataset_size=256,
    num_generations=2,
    learning_rate=2e-5,
    lora_rank=16,
    lora_alpha=32,
    lora_dropout=0.05,
    max_seq_length=2048,
    seed=17,
    env_base_url="https://your-name-your-space.hf.space",
)

if not args.env_base_url:
    raise ValueError("Set args.env_base_url to your deployed OpenEnv URL.")

args.output_dir.mkdir(parents=True, exist_ok=True)
args

## 2. Self-Contained Remote Env Client and Helper Functions

In [ ]:
@dataclass
class DdiCandidate:
    interaction_id: str
    drug_a: str
    drug_b: str
    severity: str
    evidence: str


@dataclass
class SubstitutionOption:
    regimen_id: str
    replace_drug: str
    with_drug: str
    target_condition: str
    expected_risk_delta: float
    rationale: str


@dataclass
class DdiAction:
    action_type: str
    interaction_id: Optional[str] = None
    suggested_regimen_id: Optional[str] = None
    rationale: str = ""

    def __post_init__(self) -> None:
        valid = {"flag_interaction", "monitor", "suggest_alternative", "ignore", "finish"}
        if self.action_type not in valid:
            raise ValueError(f"Invalid action_type: {self.action_type}")
        if self.action_type in {"flag_interaction", "monitor", "ignore"} and not self.interaction_id:
            raise ValueError("interaction_id is required for triage actions")
        if self.action_type == "suggest_alternative" and not self.suggested_regimen_id:
            raise ValueError("suggested_regimen_id is required for suggest_alternative")
        if self.action_type == "finish":
            self.interaction_id = None
            self.suggested_regimen_id = None


@dataclass
class DdiObservation:
    task_level: str
    task_title: str
    objective: str
    patient_id: str
    age: int
    medications: list[str]
    diagnoses: list[str]
    labs: dict[str, float]
    ddi_candidates: list[DdiCandidate] = field(default_factory=list)
    substitution_options: list[SubstitutionOption] = field(default_factory=list)
    decision_log: list[str] = field(default_factory=list)
    remaining_critical_ddis: int = 0
    current_risk_score: float = 0.0
    step_budget: int = 0
    steps_used: int = 0
    final_score: Optional[float] = None
    done: bool = False
    reward: Optional[float] = None
    metadata: dict[str, Any] = field(default_factory=dict)


class EnvAdapter(Protocol):
    def reset(self) -> DdiObservation:
        ...

    def step(self, action: DdiAction) -> DdiObservation:
        ...

    def close(self) -> None:
        ...


def _to_candidate(item: dict[str, Any]) -> DdiCandidate:
    return DdiCandidate(
        interaction_id=str(item.get("interaction_id", "")),
        drug_a=str(item.get("drug_a", "")),
        drug_b=str(item.get("drug_b", "")),
        severity=str(item.get("severity", "minor")),
        evidence=str(item.get("evidence", "")),
    )


def _to_option(item: dict[str, Any]) -> SubstitutionOption:
    return SubstitutionOption(
        regimen_id=str(item.get("regimen_id", "")),
        replace_drug=str(item.get("replace_drug", "")),
        with_drug=str(item.get("with_drug", "")),
        target_condition=str(item.get("target_condition", "")),
        expected_risk_delta=float(item.get("expected_risk_delta", 0.0)),
        rationale=str(item.get("rationale", "")),
    )


def _to_observation(payload: dict[str, Any]) -> DdiObservation:
    obs_data = payload.get("observation", payload)
    candidates = [_to_candidate(item) for item in obs_data.get("ddi_candidates", [])]
    options = [_to_option(item) for item in obs_data.get("substitution_options", [])]

    done_val = bool(payload.get("done", obs_data.get("done", False)))
    reward_val = payload.get("reward", obs_data.get("reward"))

    return DdiObservation(
        task_level=str(obs_data.get("task_level", "easy")),
        task_title=str(obs_data.get("task_title", "")),
        objective=str(obs_data.get("objective", "")),
        patient_id=str(obs_data.get("patient_id", "")),
        age=int(obs_data.get("age", 0)),
        medications=list(obs_data.get("medications", [])),
        diagnoses=list(obs_data.get("diagnoses", [])),
        labs=dict(obs_data.get("labs", {})),
        ddi_candidates=candidates,
        substitution_options=options,
        decision_log=list(obs_data.get("decision_log", [])),
        remaining_critical_ddis=int(obs_data.get("remaining_critical_ddis", 0)),
        current_risk_score=float(obs_data.get("current_risk_score", 0.0)),
        step_budget=int(obs_data.get("step_budget", 0)),
        steps_used=int(obs_data.get("steps_used", 0)),
        final_score=(None if obs_data.get("final_score") is None else float(obs_data.get("final_score"))),
        done=done_val,
        reward=(None if reward_val is None else float(reward_val)),
        metadata=dict(obs_data.get("metadata", {}) or {}),
    )


class RemoteEnvAdapter:
    def __init__(self, base_url: str, timeout_s: float = 45.0) -> None:
        self.base_url = base_url.rstrip("/")
        self.timeout_s = timeout_s
        self._session = requests.Session()

    def _post(self, endpoint: str, payload: Optional[dict[str, Any]] = None) -> dict[str, Any]:
        url = f"{self.base_url}/{endpoint.lstrip('/')}"
        response = self._session.post(url, json=(payload or {}), timeout=self.timeout_s)
        response.raise_for_status()
        return response.json()

    def reset(self) -> DdiObservation:
        return _to_observation(self._post("reset"))

    def step(self, action: DdiAction) -> DdiObservation:
        payload = {
            "action_type": action.action_type,
            "interaction_id": action.interaction_id,
            "suggested_regimen_id": action.suggested_regimen_id,
            "rationale": action.rationale,
        }
        return _to_observation(self._post("step", payload))

    def close(self) -> None:
        self._session.close()


def create_environment(args: argparse.Namespace) -> EnvAdapter:
    if not args.env_base_url:
        raise ValueError("env_base_url is required for this remote notebook")
    return RemoteEnvAdapter(args.env_base_url)


def initialize_model_and_tokenizer(args: argparse.Namespace, fast_language_model: Any):
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model, _ = fast_language_model.from_pretrained(
        model_name=args.model_name,
        max_seq_length=args.max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
    model = fast_language_model.get_peft_model(
        model,
        r=args.lora_rank,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        target_modules=LORA_TARGET_MODULES,
        bias="none",
        use_gradient_checkpointing=True,
    )
    return model, tokenizer


def build_dataset(args: argparse.Namespace) -> Dataset:
    if args.sft_dataset.exists():
        ds = load_dataset("json", data_files=str(args.sft_dataset), split="train")
        if "messages" in ds.column_names:
            prompts: list[str] = []
            for row in ds:
                messages = row.get("messages") or []
                user_prompt = next(
                    (
                        message.get("content", "")
                        for message in messages
                        if message.get("role") == "user"
                    ),
                    DEFAULT_DATASET_PROMPT,
                )
                prompts.append(user_prompt)
            if prompts:
                return Dataset.from_dict({"prompt": prompts[: args.dataset_size]})

    return Dataset.from_dict({"prompt": [DEFAULT_DATASET_PROMPT] * args.dataset_size})


def observation_to_prompt(observation: DdiObservation, history: list[str]) -> str:
    payload = {
        "task_level": observation.task_level,
        "objective": observation.objective,
        "patient_id": observation.patient_id,
        "age": observation.age,
        "labs": observation.labs,
        "diagnoses": observation.diagnoses,
        "medications": observation.medications,
        "remaining_critical_ddis": observation.remaining_critical_ddis,
        "current_risk_score": observation.current_risk_score,
        "steps_used": observation.steps_used,
        "step_budget": observation.step_budget,
        "ddi_candidates": [vars(item) for item in observation.ddi_candidates],
        "substitution_options": [vars(item) for item in observation.substitution_options],
        "history_tail": history[-4:],
    }
    return json.dumps(payload, indent=2)


def make_user_prompt(dataset_prompt: str, observation: DdiObservation) -> str:
    env_prompt = observation_to_prompt(observation, history=[])
    base_instruction = dataset_prompt.strip() or DEFAULT_DATASET_PROMPT
    return (
        f"Task instruction:\n{base_instruction}\n\n"
        f"Current patient context:\n{env_prompt}\n\n"
        "Output exactly one JSON action object."
    )


def heuristic_action(
    observation: DdiObservation,
    decided_interactions: Optional[set[str]] = None,
    suggested_regimens: Optional[set[str]] = None,
) -> dict[str, Any]:
    decisions = observation.metadata.get("decisions", {}) if observation.metadata else {}
    decided = set(decisions.keys())
    if decided_interactions:
        decided |= decided_interactions

    suggested = set(observation.metadata.get("suggested_regimens", [])) if observation.metadata else set()
    if suggested_regimens:
        suggested |= suggested_regimens

    for candidate in observation.ddi_candidates:
        if candidate.interaction_id in decided:
            continue
        if candidate.severity in {"contraindicated", "major"}:
            return {
                "action_type": "flag_interaction",
                "interaction_id": candidate.interaction_id,
                "suggested_regimen_id": None,
                "rationale": "severe interaction",
            }
        if (
            observation.task_level in {"medium", "hard"}
            and candidate.severity == "moderate"
            and (observation.age >= 80 or observation.labs.get("egfr", 90.0) < 45)
        ):
            return {
                "action_type": "flag_interaction",
                "interaction_id": candidate.interaction_id,
                "suggested_regimen_id": None,
                "rationale": "risk-amplified moderate interaction",
            }
        if candidate.severity == "moderate":
            return {
                "action_type": "monitor",
                "interaction_id": candidate.interaction_id,
                "suggested_regimen_id": None,
                "rationale": "moderate interaction monitor",
            }
        return {
            "action_type": "ignore",
            "interaction_id": candidate.interaction_id,
            "suggested_regimen_id": None,
            "rationale": "low impact",
        }

    if observation.task_level == "hard":
        for option in sorted(
            observation.substitution_options,
            key=lambda item: item.expected_risk_delta,
            reverse=True,
        ):
            if option.regimen_id in suggested:
                continue
            if option.expected_risk_delta >= HARD_REGIMEN_DELTA_THRESHOLD:
                return {
                    "action_type": "suggest_alternative",
                    "interaction_id": None,
                    "suggested_regimen_id": option.regimen_id,
                    "rationale": "high risk reduction",
                }

    return {
        "action_type": "finish",
        "interaction_id": None,
        "suggested_regimen_id": None,
        "rationale": "triage complete",
    }


def _expected_treatment_action(observation: DdiObservation, interaction_id: str) -> Optional[str]:
    for candidate in observation.ddi_candidates:
        if candidate.interaction_id != interaction_id:
            continue
        if candidate.severity in {"contraindicated", "major"}:
            return "flag_interaction"
        if (
            observation.task_level in {"medium", "hard"}
            and candidate.severity == "moderate"
            and (observation.age >= 80 or observation.labs.get("egfr", 90.0) < 45)
        ):
            return "flag_interaction"
        if candidate.severity == "moderate":
            return "monitor"
        return "ignore"
    return None


def apply_action_guardrails(
    payload: dict[str, Any],
    observation: DdiObservation,
    decided_interactions: Optional[set[str]] = None,
    suggested_regimens: Optional[set[str]] = None,
) -> dict[str, Any]:
    action_type = payload.get("action_type")
    interaction_id = payload.get("interaction_id")
    suggested_regimen_id = payload.get("suggested_regimen_id")

    metadata_decisions = observation.metadata.get("decisions", {}) if observation.metadata else {}
    all_decided = set(metadata_decisions.keys())
    if decided_interactions:
        all_decided |= decided_interactions

    metadata_suggested = set(observation.metadata.get("suggested_regimens", [])) if observation.metadata else set()
    if suggested_regimens:
        metadata_suggested |= suggested_regimens

    unresolved = {item.interaction_id for item in observation.ddi_candidates if item.interaction_id not in all_decided}

    pending_regimens: list[str] = []
    if observation.task_level == "hard":
        options = sorted(observation.substitution_options, key=lambda item: item.expected_risk_delta, reverse=True)
        pending_regimens = [
            item.regimen_id
            for item in options
            if item.regimen_id not in metadata_suggested and item.expected_risk_delta >= HARD_REGIMEN_DELTA_THRESHOLD
        ]

    if action_type in {"flag_interaction", "monitor", "ignore"}:
        if interaction_id not in unresolved:
            return heuristic_action(observation, all_decided, metadata_suggested)
        expected = _expected_treatment_action(observation, interaction_id)
        if expected is None:
            return heuristic_action(observation, all_decided, metadata_suggested)
        if action_type != expected:
            return {
                "action_type": expected,
                "interaction_id": interaction_id,
                "suggested_regimen_id": None,
                "rationale": "guardrail corrected triage",
            }
        return {
            "action_type": action_type,
            "interaction_id": interaction_id,
            "suggested_regimen_id": None,
            "rationale": payload.get("rationale", ""),
        }

    if action_type == "suggest_alternative":
        valid_option_ids = {item.regimen_id for item in observation.substitution_options}
        if observation.task_level != "hard" or not suggested_regimen_id:
            return heuristic_action(observation, all_decided, metadata_suggested)
        if suggested_regimen_id in metadata_suggested:
            return heuristic_action(observation, all_decided, metadata_suggested)
        if suggested_regimen_id not in valid_option_ids:
            return heuristic_action(observation, all_decided, metadata_suggested)
        if pending_regimens and suggested_regimen_id not in pending_regimens:
            return heuristic_action(observation, all_decided, metadata_suggested)
        return {
            "action_type": "suggest_alternative",
            "interaction_id": None,
            "suggested_regimen_id": suggested_regimen_id,
            "rationale": payload.get("rationale", ""),
        }

    if action_type == "finish":
        if unresolved:
            return heuristic_action(observation, all_decided, metadata_suggested)
        if observation.task_level == "hard" and pending_regimens:
            return heuristic_action(observation, all_decided, metadata_suggested)
        return {
            "action_type": "finish",
            "interaction_id": None,
            "suggested_regimen_id": None,
            "rationale": payload.get("rationale", ""),
        }

    return heuristic_action(observation, all_decided, metadata_suggested)


def parse_action(content: str, observation: DdiObservation) -> dict[str, Any]:
    if not content:
        return heuristic_action(observation)

    stripped = content.strip()
    try:
        parsed = json.loads(stripped)
    except json.JSONDecodeError:
        match = JSON_BLOCK.search(stripped)
        if not match:
            return heuristic_action(observation)
        try:
            parsed = json.loads(match.group(0))
        except json.JSONDecodeError:
            return heuristic_action(observation)

    candidate = {
        "action_type": parsed.get("action_type"),
        "interaction_id": parsed.get("interaction_id"),
        "suggested_regimen_id": parsed.get("suggested_regimen_id"),
        "rationale": parsed.get("rationale", ""),
    }
    try:
        DdiAction(**candidate)
        return candidate
    except Exception:
        return heuristic_action(observation)


def extract_component_reward(observation: DdiObservation) -> dict[str, float]:
    metadata = observation.metadata or {}
    components = metadata.get("reward_components", {})
    return {
        "triage_reward": float(components.get("triage_score", 0.0)),
        "regimen_reward": float(components.get("regimen_score", 0.0)),
        "risk_delta_reward": float(components.get("risk_delta_bonus", 0.0)),
        "invalid_penalty": float(components.get("invalid_action_penalty", 0.0)),
        "final_score_reward": float(observation.final_score or 0.0),
    }


def warmup_with_heuristic(env: EnvAdapter, warmup_steps: int) -> list[float]:
    observation = env.reset()
    reward_trace: list[float] = []
    decided_interactions: set[str] = set()
    suggested_regimens: set[str] = set()

    for _ in range(max(1, warmup_steps)):
        payload = heuristic_action(
            observation,
            decided_interactions=decided_interactions,
            suggested_regimens=suggested_regimens,
        )
        payload = apply_action_guardrails(
            payload,
            observation,
            decided_interactions=decided_interactions,
            suggested_regimens=suggested_regimens,
        )

        action = DdiAction(**payload)
        if action.interaction_id:
            decided_interactions.add(action.interaction_id)
        if action.suggested_regimen_id:
            suggested_regimens.add(action.suggested_regimen_id)

        observation = env.step(action)
        reward_trace.append(float(observation.reward or 0.0))

        if observation.done:
            observation = env.reset()
            decided_interactions.clear()
            suggested_regimens.clear()

    return reward_trace

## 3. Rollout and Reward Functions

In [ ]:
def rollout_once(
    *,
    trainer: Any,
    env: EnvAdapter,
    tokenizer: Any,
    dataset_prompt: str,
    max_turns: int,
    generate_rollout_completions: Callable[..., Any],
) -> dict[str, Any]:
    observation = env.reset()

    prompt_ids: list[int] = []
    completion_ids: list[int] = []
    logprobs: list[float] = []

    triage_rewards: list[float] = []
    regimen_rewards: list[float] = []
    risk_delta_rewards: list[float] = []
    invalid_penalties: list[float] = []
    final_score_rewards: list[float] = []

    decided_interactions: set[str] = set()
    suggested_regimens: set[str] = set()

    for _ in range(max_turns):
        if observation.done:
            break

        user_prompt = make_user_prompt(dataset_prompt, observation)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        )

        rollout_outputs = generate_rollout_completions(trainer, [prompt_text])[0]

        prompt_ids.extend(rollout_outputs.get("prompt_ids", []))
        completion_ids.extend(rollout_outputs.get("completion_ids", []))
        logprobs.extend(rollout_outputs.get("logprobs", []))

        completion_text = rollout_outputs.get("text")
        if not completion_text:
            completion_text = tokenizer.decode(
                rollout_outputs.get("completion_ids", []),
                skip_special_tokens=True,
            )

        raw_payload = parse_action(completion_text, observation)
        payload = apply_action_guardrails(
            raw_payload,
            observation,
            decided_interactions=decided_interactions,
            suggested_regimens=suggested_regimens,
        )

        action = DdiAction(**payload)
        if action.interaction_id:
            decided_interactions.add(action.interaction_id)
        if action.suggested_regimen_id:
            suggested_regimens.add(action.suggested_regimen_id)

        observation = env.step(action)
        component_reward = extract_component_reward(observation)

        triage_rewards.append(component_reward["triage_reward"])
        regimen_rewards.append(component_reward["regimen_reward"])
        risk_delta_rewards.append(component_reward["risk_delta_reward"])
        invalid_penalties.append(component_reward["invalid_penalty"])
        final_score_rewards.append(component_reward["final_score_reward"])

    return {
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "logprobs": logprobs,
        "triage_reward": triage_rewards[-1] if triage_rewards else 0.0,
        "regimen_reward": regimen_rewards[-1] if regimen_rewards else 0.0,
        "risk_delta_reward": risk_delta_rewards[-1] if risk_delta_rewards else 0.0,
        "invalid_penalty": invalid_penalties[-1] if invalid_penalties else 0.0,
        "final_score_reward": final_score_rewards[-1] if final_score_rewards else 0.0,
    }


def build_rollout_func(
    *,
    env: EnvAdapter,
    tokenizer: Any,
    max_turns: int,
    generate_rollout_completions: Callable[..., Any],
):
    def rollout_func(prompts, trainer=None):
        if trainer is None:
            raise RuntimeError("GRPO rollout_func received no trainer instance.")

        episode_prompt_ids = []
        episode_completion_ids = []
        episode_logprobs = []
        triage_rewards = []
        regimen_rewards = []
        risk_delta_rewards = []
        invalid_penalties = []
        final_score_rewards = []

        for dataset_prompt in prompts:
            episode = rollout_once(
                trainer=trainer,
                env=env,
                tokenizer=tokenizer,
                dataset_prompt=dataset_prompt,
                max_turns=max_turns,
                generate_rollout_completions=generate_rollout_completions,
            )

            episode_prompt_ids.append(episode["prompt_ids"])
            episode_completion_ids.append(episode["completion_ids"])
            episode_logprobs.append(episode["logprobs"])
            triage_rewards.append(episode["triage_reward"])
            regimen_rewards.append(episode["regimen_reward"])
            risk_delta_rewards.append(episode["risk_delta_reward"])
            invalid_penalties.append(episode["invalid_penalty"])
            final_score_rewards.append(episode["final_score_reward"])

        return {
            "prompt_ids": episode_prompt_ids,
            "completion_ids": episode_completion_ids,
            "logprobs": episode_logprobs,
            "triage_reward": triage_rewards,
            "regimen_reward": regimen_rewards,
            "risk_delta_reward": risk_delta_rewards,
            "invalid_penalty": invalid_penalties,
            "final_score_reward": final_score_rewards,
        }

    return rollout_func


def _reward_vector(reward_key: str, completions: list[Any], kwargs: dict[str, Any]) -> list[float]:
    rewards = kwargs.get(reward_key)
    if rewards:
        return [float(r) for r in rewards]
    return [0.0] * len(completions)


def reward_triage(completions, **kwargs):
    return _reward_vector("triage_reward", completions, kwargs)


def reward_regimen(completions, **kwargs):
    return _reward_vector("regimen_reward", completions, kwargs)


def reward_risk_delta(completions, **kwargs):
    return _reward_vector("risk_delta_reward", completions, kwargs)


def reward_invalid_penalty(completions, **kwargs):
    return _reward_vector("invalid_penalty", completions, kwargs)


def reward_final_score(completions, **kwargs):
    return _reward_vector("final_score_reward", completions, kwargs)


def build_grpo_config(args: argparse.Namespace, grpo_config_cls: Any):
    return grpo_config_cls(
        output_dir=str(args.output_dir),
        learning_rate=args.learning_rate,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=max(1, min(20, args.train_steps // 6)),
        num_generations=args.num_generations,
        max_completion_length=96,
        logging_steps=5,
        save_steps=max(20, args.train_steps // 3),
        max_steps=args.train_steps,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        bf16=False,
        fp16=True,
        seed=args.seed,
    )


def save_artifacts(
    *,
    output_dir: Path,
    trainer: Any,
    tokenizer: Any,
    stats: Any,
    warmup_reward_trace: list[float],
) -> None:
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))

    (output_dir / "warmup_reward_trace.json").write_text(
        json.dumps({"warmup_rewards": warmup_reward_trace}, indent=2),
        encoding="utf-8",
    )

    metrics = stats.metrics if hasattr(stats, "metrics") else {}
    (output_dir / "train_metrics.json").write_text(
        json.dumps(metrics, indent=2),
        encoding="utf-8",
    )

## 4. Initialize Model, Environment, and Trainer

In [ ]:
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from trl.experimental.openenv import generate_rollout_completions

model, tokenizer = initialize_model_and_tokenizer(args, FastLanguageModel)
env = create_environment(args)

warmup_reward_trace = warmup_with_heuristic(env, args.warmup_steps)
dataset = build_dataset(args)

rollout_func = build_rollout_func(
    env=env,
    tokenizer=tokenizer,
    max_turns=args.max_turns,
    generate_rollout_completions=generate_rollout_completions,
)

grpo_config = build_grpo_config(args, GRPOConfig)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        reward_triage,
        reward_regimen,
        reward_risk_delta,
        reward_invalid_penalty,
        reward_final_score,
    ],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
)

len(dataset)

## 5. Train and Save

Run this cell to start training. It saves model, tokenizer, warmup trace, and metrics.

In [ ]:
try:
    stats = trainer.train()
    save_artifacts(
        output_dir=args.output_dir,
        trainer=trainer,
        tokenizer=tokenizer,
        stats=stats,
        warmup_reward_trace=warmup_reward_trace,
    )
    print(f"Training complete. Artifacts saved to: {args.output_dir}")
finally:
    env.close()